# Car Price Prediction with Machine Learning
**Oasis Infobyte — Data Science Internship | Task 3**  
**Project:** Predicting used-car selling prices  
**Author:** Sumit Kumar

## Objective
Build and compare regression models that estimate a used car's selling price from its available characteristics. This notebook performs data inspection, cleaning, exploratory analysis, feature engineering, model training, evaluation, and feature-importance analysis.

> Important: model scores and findings are calculated when you run the notebook. They are not hard-coded or invented.

## 1. Import libraries
We use Pandas/NumPy for data handling, Matplotlib/Seaborn for visualizations, and scikit-learn for preprocessing and regression models.

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 2. Load the dataset
The project uses the commonly used CarDekho used-car dataset with fields such as `Car_Name`, `Year`, `Selling_Price`, `Present_Price`, `Kms_Driven`, `Fuel_Type`, `Seller_Type`, `Transmission`, and `Owner`.

The notebook first looks for a CSV in `dataset/` or the project folder. If none is found, it attempts to download a public copy. If download fails, download the CSV manually from the dataset source linked in the README and place it in `dataset/`.

In [ ]:
DATA_DIR = Path("dataset")
DATA_DIR.mkdir(exist_ok=True)

candidate_files = [
    DATA_DIR / "CarDekho_Dataset.csv",
    DATA_DIR / "car_data.csv",
    Path("CarDekho_Dataset.csv"),
    Path("car_data.csv"),
]

csv_path = next((p for p in candidate_files if p.exists()), None)

if csv_path is None:
    dataset_url = (
        "https://raw.githubusercontent.com/"
        "pujarisurakshashankar/car-market-trend-analysis/main/"
        "data/CarDekho_Dataset.csv"
    )
    download_path = DATA_DIR / "CarDekho_Dataset.csv"
    try:
        urllib.request.urlretrieve(dataset_url, download_path)
        csv_path = download_path
        print(f"Dataset downloaded to: {csv_path}")
    except Exception as exc:
        raise FileNotFoundError(
            "Dataset was not found locally and automatic download failed. "
            "Download the CSV described in README.md and place it in the dataset/ folder."
        ) from exc

df = pd.read_csv(csv_path)
print("Dataset file:", csv_path)
print("Shape:", df.shape)
display(df.head())

## 3. Inspect data quality
Check columns, data types, missing values, duplicates, and summary statistics before cleaning.

In [ ]:
print("Column names:", df.columns.tolist())
display(df.info())
display(df.describe(include="all").T)
print("\\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())

## 4. Clean and standardize columns
Column names may vary slightly between copies of the dataset, so standardize common alternatives. We only remove duplicate rows and let model pipelines impute missing values where needed.

In [ ]:
# Normalize column names and common naming variations.
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]
rename_map = {
    "Car_Name": "Car_Name", "name": "Car_Name", "Name": "Car_Name",
    "year": "Year", "selling_price": "Selling_Price",
    "present_price": "Present_Price", "km_driven": "Kms_Driven",
    "kms_driven": "Kms_Driven", "fuel": "Fuel_Type",
    "seller_type": "Seller_Type", "transmission": "Transmission",
    "owner": "Owner"
}
df = df.rename(columns={c: rename_map[c] for c in df.columns if c in rename_map})

required = ["Year", "Selling_Price"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}. Available columns: {df.columns.tolist()}")

# Convert known numeric fields safely when they exist.
for col in ["Year", "Selling_Price", "Present_Price", "Kms_Driven", "Owner"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Car_Name", "Fuel_Type", "Seller_Type", "Transmission"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

before = len(df)
df = df.drop_duplicates().copy()
print(f"Removed {before - len(df)} duplicate rows.")
print("Cleaned shape:", df.shape)
display(df.head())

## 5. Feature engineering
Calculate `Car_Age` from the manufacturing year. The reference year is the current calendar year when the notebook runs. The car name is retained as a categorical feature if available; the pipeline handles categories with one-hot encoding.

In [ ]:
reference_year = pd.Timestamp.today().year
df["Car_Age"] = reference_year - df["Year"]
df.loc[df["Car_Age"] < 0, "Car_Age"] = np.nan

# Remove rows with an unavailable or invalid target; the target must be known for training.
df = df.dropna(subset=["Selling_Price"]).copy()
df = df[df["Selling_Price"] >= 0].copy()

print("Reference year:", reference_year)
display(df[["Year", "Car_Age", "Selling_Price"]].head())

## 6. Exploratory Data Analysis (EDA)
Charts below are generated from the actual dataset after basic cleaning.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Selling_Price", kde=True)
plt.title("Distribution of Used-Car Selling Prices")
plt.xlabel("Selling price (dataset units)")
plt.ylabel("Number of listings")
plt.tight_layout()
plt.show()

In [ ]:
if "Fuel_Type" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="Fuel_Type", y="Selling_Price")
    plt.title("Selling Price by Fuel Type")
    plt.xlabel("Fuel type")
    plt.ylabel("Selling price (dataset units)")
    plt.tight_layout()
    plt.show()
else:
    print("Fuel_Type is not available in this dataset.")

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Car_Age", y="Selling_Price", alpha=0.7)
plt.title("Selling Price vs. Car Age")
plt.xlabel("Car age (years)")
plt.ylabel("Selling price (dataset units)")
plt.tight_layout()
plt.show()

In [ ]:
if "Kms_Driven" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Kms_Driven", y="Selling_Price", alpha=0.7)
    plt.title("Selling Price vs. Kilometres Driven")
    plt.xlabel("Kilometres driven")
    plt.ylabel("Selling price (dataset units)")
    plt.tight_layout()
    plt.show()

## 7. Correlation heatmap
This heatmap uses numeric columns only. Correlation indicates association, not causation.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(9, 6))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
    plt.title("Correlation Heatmap of Numeric Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation heatmap.")

## 8. Prepare features and split data
`Selling_Price` is the target. We exclude the target from the input features. We also exclude `Year` because its information is represented by engineered `Car_Age`. Missing numerical values are median-imputed; categorical values are most-frequent-imputed and one-hot encoded. The preprocessing is fitted only on training data through a Pipeline to reduce data leakage.

In [ ]:
target = "Selling_Price"
drop_from_features = [target]
if "Year" in df.columns:
    drop_from_features.append("Year")

X = df.drop(columns=drop_from_features)
y = df[target]

# Remove any completely empty columns; they cannot contribute useful information.
X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Features:", X.columns.tolist())
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 9. Train and compare regression models
We compare Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. The best model is chosen using the lowest test-set RMSE, while MAE and R² are also reported.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE)
}

trained_models = {}
results = []

for name, estimator in models.items():
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    trained_models[name] = model
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(3))
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
print("Best model by test RMSE:", best_model_name)

## 10. Actual vs. predicted values
A closer alignment with the diagonal line indicates better predictions. This chart uses the best model selected above.

In [ ]:
best_predictions = best_model.predict(X_test)
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_predictions, alpha=0.75)
low = min(float(y_test.min()), float(np.min(best_predictions)))
high = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([low, high], [low, high], linestyle="--")
plt.title(f"Actual vs. Predicted Selling Price — {best_model_name}")
plt.xlabel("Actual selling price")
plt.ylabel("Predicted selling price")
plt.tight_layout()
plt.show()

## 11. Residual analysis
Residuals are actual values minus predicted values. Patterns in residuals can indicate where the model performs poorly.

In [ ]:
residuals = y_test - best_predictions
plt.figure(figsize=(8, 5))
sns.scatterplot(x=best_predictions, y=residuals, alpha=0.75)
plt.axhline(0, linestyle="--")
plt.title(f"Residual Plot — {best_model_name}")
plt.xlabel("Predicted selling price")
plt.ylabel("Residual (actual - predicted)")
plt.tight_layout()
plt.show()

## 12. Feature importance using permutation importance
Permutation importance measures how much the model score drops when a feature is shuffled. It works with the complete pipeline and avoids relying on model-specific importance scales. Computation may take a little time.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error"
)
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(importance_df.head(15).round(4))

top_features = importance_df.head(15).sort_values("Importance_Mean")
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"].astype(str), top_features["Importance_Mean"], xerr=top_features["Importance_Std"])
plt.title("Top Features by Permutation Importance")
plt.xlabel("Decrease in model score when feature is shuffled")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 13. Generate evidence-based summary
The following prints metrics from the actual run. Use these values when writing the final report; do not copy example scores from other projects.

In [ ]:
best_row = results_df.iloc[0]
print("PROJECT RESULTS")
print("-" * 50)
print(f"Rows used: {len(df)}")
print(f"Best model by RMSE: {best_row['Model']}")
print(f"MAE:  {best_row['MAE']:.3f}")
print(f"RMSE: {best_row['RMSE']:.3f}")
print(f"R²:   {best_row['R2']:.3f}")
print("\\nInterpretation:")
print("- MAE is the average absolute prediction error in the target's units.")
print("- RMSE penalizes larger prediction errors more strongly.")
print("- R² describes how much variation in selling price is explained by the model; it can be negative on a test set.")
print("- These metrics are estimates from one train/test split, not a guarantee of future-market accuracy.")

## 14. Conclusion and limitations
Write the final conclusion after running all cells. Mention the best model and actual metrics printed above. Explain which features mattered most according to permutation importance.

### Limitations
- The dataset may be small or dated and may not represent today's entire used-car market.
- The predictions reflect the dataset's price units and collection context.
- Important factors such as city, vehicle condition, insurance, variant, and current market demand may be unavailable.
- A random train/test split gives one estimate; cross-validation and external validation would provide a stronger assessment.
- Correlation and feature importance do not prove causation.

## 15. Reproducibility
For repeatable results, the train/test split and model random states are fixed. Re-run all cells from the top if the dataset changes.